# ESC-50: complete development matrix / 完整开发实验矩阵

This notebook executes the supplied 65-task matrix on Colab. / 本 notebook 在 Colab 实际执行所提供的65项任务。

Main experiments use 60 epochs and seeds 101/202/303; separate two-epoch pilot results are excluded. Results are development-validation measurements, not independent confirmation. / 正式实验60轮；2轮试跑独立保存，不纳入正式结果。这些是开发验证结果，不是独立确认。

Storage in this execution uses the Colab local disk and an atomic export ZIP after every completed task, downloaded to the local project during execution. No Drive mount or paid upgrade is needed. / 本次使用Colab本地磁盘，每项任务完成后导出ZIP并及时下载，无需新增Drive挂载或付费升级。


## 1 · User settings (edit only this cell to start)

In [ ]:
# Default plan-only: NO download, NO Drive prompt, NO training.
RUN_MODE = "selected"                  # "plan" | "pilot" | "selected"
GROUPS_TO_RUN = ["A_baselines", "B_architecture", "C_augmentation", "D_feature_sampling", "E_robustness"]    # e.g. ["B_architecture", "C_augmentation"]
TASK_IDS = []                       # optional exact task IDs; when nonempty overrides groups
MAX_TASKS_PER_SESSION = 65           # deliberate Colab safety cap (raise explicitly)
USE_GOOGLE_DRIVE = False             # persistent results; mounted only for actual execution
STUDY_MATRIX_PATH = ""              # optional supplied JSON; otherwise bundled config
PROJECT_ZIP_PATH = ""               # optional existing ZIP on /content or in Drive
DATASET_ZIP_PATH = ""               # optional official ESC-50 zip on /content or Drive
PERSIST_BASE_CACHE = False
PERSIST_FEATURE_CACHES = False      # 12 caches use significant space; regeneration is allowed
PILOT_EPOCHS = 2                    # separate pilot namespace; never called final evidence
FULL_EPOCHS = 60
BATCH_SIZE = 32
LEARNING_RATE = 0.001
DEVICE = "cuda"                     # GPU if available, otherwise CPU
CPU_THREADS = 2
MAX_SECONDS_PER_TRAIN = 600       # CLI training time limit; can resume main runs
MAX_SESSION_MINUTES = 120           # stop starting new tasks after this budget
DRIVE_SUBFOLDER = "ESC50_Research_Colab"
assert RUN_MODE in {"plan", "pilot", "selected"}
assert DEVICE in {"auto", "cpu", "cuda"}
assert MAX_TASKS_PER_SESSION >= 1 and FULL_EPOCHS >= 1 and PILOT_EPOCHS >= 1

## 2 · Preview the full experiment matrix

In [ ]:
import json, pathlib, os, sys, time, shutil, subprocess
from collections import Counter
from IPython.display import display

# Exact 12-case roster is also recorded in the standalone study-matrix JSON.
FEATURES = ("logmel64", "logmel32", "mfcc40", "loglinear64")
FEATURE_CASES = [f"sr{sr}_fmax{fmax}_{feat}" for sr, fmax in ((16, 8), (32, 8), (32, 16)) for feat in FEATURES]
SEEDS = (101, 202, 303)

def build_task_list():
    tasks = [dict(id="A_logistic", group="A_baselines", kind="baseline", method="logistic"),
             dict(id="A_cepstral_svm", group="A_baselines", kind="baseline", method="cepstral_svm")]
    for model in ("cnn_freq", "crnn_freq", "crnn_freq_ta", "crnn_freq_ca", "crnn_freq_tca"):
        for seed in SEEDS:
            tasks.append(dict(id=f"B_{model}_s{seed}",group="B_architecture",kind="supervised", model=model, seed=seed, augmentation="clean"))
    for augmentation in ("specaugment", "noise", "mixup"):
        for seed in SEEDS:
            tasks.append(dict(id=f"C_{augmentation}_s{seed}", group="C_augmentation",kind="supervised", model="cnn_freq",seed=seed,augmentation=augmentation))
    for case in FEATURE_CASES:
        for seed in SEEDS:
            tasks.append(dict(id=f"D_{case}_s{seed}",group="D_feature_sampling",kind="feature_ablation",case=case,model="cnn",seed=seed))
    for name, dependency in (("clean_cnn", "B_cnn_freq_s101"), ("clean_crnn", "B_crnn_freq_s101"), ("noise_cnn", "C_noise_s101")):
        tasks.append(dict(id=f"E_{name}",group="E_robustness",kind="robustness",checkpoint_task=dependency, conditions=["clean","snr_20","snr_10","snr_0","gain_-6","gain_6"]))
    return tasks

def checked_matrix(matrix):
    if matrix.get('schema') != 'esc50_colab_study_v1':
        raise ValueError('Unsupported study matrix schema')
    protocol = matrix.get('train_protocol', {})
    fixed = dict(train_folds=[1,2,3], validation_fold=4, held_out_fold=5,
                 source_strict=True, seeds=list(SEEDS), scheduler='cosine')
    if any(protocol.get(k) != v for k,v in fixed.items()):
        raise ValueError('Study matrix changes the supported development protocol')
    settings = dict(epochs=FULL_EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE)
    if any(protocol.get(k) != v for k,v in settings.items()):
        raise ValueError('Notebook full-run settings disagree with study matrix')
    reference = {t['id']:t for t in build_task_list()}
    tasks = matrix.get('tasks', [])
    if len(tasks) != len(reference) or len({t.get('id') for t in tasks}) != len(tasks):
        raise ValueError('Study matrix must contain the 65 unique supported tasks')
    for task in tasks:
        expected = reference.get(task.get('id'))
        if expected is None or any(task.get(k) != v for k,v in expected.items()):
            raise ValueError(f'Study task semantics mismatch: {task.get("id")}')
    return tasks

def select_tasks(tasks):
    if TASK_IDS:
        unknown=set(TASK_IDS)-{t['id'] for t in tasks}
        if unknown: raise ValueError(f'Unknown task IDs: {sorted(unknown)}')
        return [t for t in tasks if t['id'] in TASK_IDS]
    unknown=set(GROUPS_TO_RUN)-{t['group'] for t in tasks}
    if unknown: raise ValueError(f'Unimplemented/unknown groups: {sorted(unknown)}')
    return [t for t in tasks if t['group'] in GROUPS_TO_RUN]

matrix_preview = pathlib.Path(STUDY_MATRIX_PATH).expanduser() if STUDY_MATRIX_PATH else pathlib.Path('configs/colab_study_matrix_v1.json')
if STUDY_MATRIX_PATH and not matrix_preview.is_file():
    raise FileNotFoundError('STUDY_MATRIX_PATH does not exist')
if STUDY_MATRIX_PATH:STUDY_MATRIX_PATH=str(matrix_preview.resolve())
TASKS = checked_matrix(json.loads(matrix_preview.read_text())) if matrix_preview.is_file() else build_task_list()
COUNTS = Counter(t['group'] for t in TASKS)
print("Registered executable tasks:", len(TASKS))
for group,n in COUNTS.items():print(f"  {group:<24} {n:>3}")
print("Manual / unavailable: PANNs, AST, BEATs, CLAP, RL, independent holdout and confirmation CV")
print("Default:",RUN_MODE, "| selected groups:",GROUPS_TO_RUN,"| exact IDs:",TASK_IDS)
SELECTED=select_tasks(TASKS)
print("Selected:",len(SELECTED),"| per-session cap:",MAX_TASKS_PER_SESSION)
print("First selected tasks:",[t['id'] for t in SELECTED[:12]])
assert len(TASKS)==65


## 3 · Environment and persistence (execution modes only)

**Execution budget: each subprocess, including silent feature runs, is capped by the remaining session deadline and per-command limit. A timeout preserves partial evidence for inspection. **Colab storage:**** Project code, WAVs, and feature caches run from local `/content` for speed. The study results directory is on Google Drive when enabled, keeping completed runs and main-training `last.pt` snapshots across sessions. A Colab runtime may change Python, CUDA or package versions; **the legacy resume mechanism can reject such an environment even if a checkpoint exists**, so completed runs are safe to skip but resume is only attempted when compatible.

If you prefer no Drive, set `USE_GOOGLE_DRIVE=False`; results then live on local ephemeral Colab storage and can disappear after disconnect.

In [ ]:
from pathlib import Path
import zipfile

IS_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
ROOT=None
STORAGE=None
EXECUTE_SETUP=(RUN_MODE!="plan")

def find_project(base:Path):
    paths=[base,base/'esc50-sound-classification']
    paths+=list(base.glob('*/esc50-sound-classification')) if base.exists() else []
    for p in paths:
        if (p/'pyproject.toml').is_file() and (p / 'scripts/feature_ablation.py').is_file(): return p.resolve()
    return None

if EXECUTE_SETUP:
    if IS_COLAB and USE_GOOGLE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        STORAGE=Path('/content/drive/MyDrive')/DRIVE_SUBFOLDER
    else:
        STORAGE=Path('/content/esc50_results') if IS_COLAB else Path.cwd()/'colab_local_results'
    STORAGE.mkdir(parents=True,exist_ok=True)
    for candidate in (Path.cwd(),Path.cwd().parent,Path('/content'),Path('/mnt/data/esc50_ablation_work')):
        ROOT=find_project(candidate)
        if ROOT:break
    if ROOT is None:
        known=Path(PROJECT_ZIP_PATH).expanduser() if PROJECT_ZIP_PATH else STORAGE/'ESC50_project.zip'
        if not known.is_file():
            if not IS_COLAB:
                raise FileNotFoundError('Project ZIP not found. Set PROJECT_ZIP_PATH to the full project ZIP.')
            from google.colab import files
            print('Upload ESC50_Feature_Sampling_Ablation_Full.zip (complete project)')
            selected=files.upload()
            if len(selected)!=1:raise ValueError('Upload exactly one project ZIP')
            known=Path('/content')/next(iter(selected))
            if USE_GOOGLE_DRIVE: shutil.copy2(known,STORAGE/'ESC50_project.zip')
        target=Path('/content/esc50_project') if IS_COLAB else Path.cwd()/'esc50_project'
        target.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(known) as archive:
            for item in archive.infolist():
                relative=Path(item.filename)
                if relative.is_absolute() or '..' in relative.parts:
                    raise ValueError('Unsafe ZIP member')
            archive.extractall(target)
        ROOT=find_project(target)
        if ROOT is None:raise RuntimeError('ZIP did not contain the expected ESC-50 project structure')
    os.chdir(ROOT)
    # Editable installs do not refresh sys.path in an already-running kernel.
    if str(ROOT/'src') not in sys.path:sys.path.insert(0,str(ROOT/'src'))
    subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[baseline,report]'],check=True,cwd=ROOT,timeout=300)
    import torch
    print('Project:',ROOT, '| torch:',torch.__version__, '| GPU:',torch.cuda.is_available())
    if DEVICE=='cuda' and not torch.cuda.is_available():raise RuntimeError('CUDA requested but unavailable')
    # Ensure the notebook config agrees with the code bundle used for experiments.
    matrix_path=ROOT/'configs'/'colab_study_matrix_v1.json'
    if not matrix_path.is_file():raise FileNotFoundError('Use the matching Colab-enabled full project ZIP')
    expected=json.loads(matrix_path.read_text())
    if STUDY_MATRIX_PATH:
        supplied=json.loads(Path(STUDY_MATRIX_PATH).expanduser().read_text())
        if supplied != expected:raise ValueError('Supplied study matrix differs from project configuration')
    TASKS=checked_matrix(expected)
    SELECTED=select_tasks(TASKS)
    RUNS=STORAGE / 'runs'; REPORTS=STORAGE/'reports'; LOGS=STORAGE/'logs'
    for p in (RUNS,REPORTS,LOGS):p.mkdir(parents=True,exist_ok=True)
else:
    print('PLAN MODE — setup skipped: no upload, no install, no Drive mount.')

## 4 · Data acquisition and baseline cache

In [ ]:
# Dataset download is deliberately separate from the experiment plan preview.
# The original WAV dataset is retained in Drive as an archive if enabled.

def execute_command(args, label, timeout=None):
    """Bound even silent processes; the session deadline also caps feature runs."""
    if not EXECUTE_SETUP:raise RuntimeError('Execution disabled in plan mode')
    import signal
    env=dict(os.environ, PYTHONPATH=str(ROOT/'src'), OMP_NUM_THREADS=str(CPU_THREADS), MKL_NUM_THREADS=str(CPU_THREADS))
    log_file=LOGS/(label+'.log')
    if log_file.exists():log_file=LOGS/(label+'_'+str(time.time_ns())+'.log')
    limit = MAX_SECONDS_PER_TRAIN if timeout is None else timeout
    deadline = min(time.monotonic()+limit, globals().get('_session_deadline', float('inf')))
    if deadline <= time.monotonic():raise TimeoutError('Session budget exhausted before starting subprocess')
    print('$',' '.join(str(v) for v in args),flush=True)
    started=time.monotonic()
    with log_file.open('w') as stream, log_file.open('r') as reader:
        process=subprocess.Popen([str(x) for x in args],cwd=ROOT,stdout=stream,stderr=subprocess.STDOUT,
                                 env=env,text=True,start_new_session=True)
        try:
            while process.poll() is None:
                chunk=reader.read()
                if chunk:print(chunk,end='',flush=True)
                if time.monotonic() >= deadline:raise TimeoutError(f'{label} exceeded execution/session budget; log: {log_file}')
                time.sleep(0.2)
            chunk=reader.read()
            if chunk:print(chunk,end='',flush=True)
            rc=process.returncode
        except BaseException:
            if process.poll() is None:
                os.killpg(process.pid,signal.SIGTERM)
                try:process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid,signal.SIGKILL);process.wait()
            raise
    if rc:raise RuntimeError(f'{label} exited with code {rc}; log: {log_file}')
    print('Finished in',round(time.monotonic()-started,1),'seconds | log:',log_file)

def ensure_audio():
    raw=ROOT/'data'/'ESC-50'
    meta=raw/'meta'/'esc50.csv'
    if meta.is_file() and len(list((raw/'audio').glob('*.wav')))==2000:
        return raw
    # Use the official pinned commit defined in the project. Keep ZIP to avoid
    # re-downloading after Colab disconnections when Drive storage is enabled.
    from esc50_sound.data import DATA_COMMIT
    url=f'https://github.com/karoldvl/ESC-50/archive/{DATA_COMMIT}.zip'
    saved=(STORAGE/'ESC50_official_pinned.zip') if USE_GOOGLE_DRIVE else (ROOT/'data'/'ESC50_official_pinned.zip')
    candidate=Path(DATASET_ZIP_PATH).expanduser() if DATASET_ZIP_PATH else saved
    if not candidate.exists():
        from urllib.request import Request,urlopen
        candidate.parent.mkdir(parents=True,exist_ok=True)
        print('Downloading official ESC-50 source (may take several minutes):',url)
        with urlopen(Request(url,headers={'User-Agent':'ESC50-Colab-Research'}),timeout=120) as response, candidate.open('wb') as dst:
            shutil.copyfileobj(response,dst)
    raw.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(candidate) as archive:
        for entry in archive.infolist():
            parts=Path(entry.filename).parts
            if '..' in parts or not parts:raise ValueError('Unsafe dataset archive entry')
            if len(parts)<3 or parts[1] not in {'audio','meta'}:continue
            relative=Path(*parts[1:])
            target=raw/relative
            if entry.is_dir():target.mkdir(parents=True,exist_ok=True)
            else:
                target.parent.mkdir(parents=True,exist_ok=True)
                with archive.open(entry) as src,target.open('wb') as dst:shutil.copyfileobj(src,dst)
    if not meta.is_file() or len(list((raw/'audio').glob('*.wav')))!=2000:
        raise RuntimeError('ESC-50 WAV files or official metadata incomplete after extraction')
    return raw

def ensure_base_cache():
    raw=ensure_audio()
    cache=ROOT/'data'/'esc50_features.npz'
    sidecar=Path(str(cache)+'.json')
    saved=STORAGE/'cache'/'esc50_features.npz'
    saved_sidecar=Path(str(saved)+'.json')
    if not (cache.is_file() and sidecar.is_file()):
        if PERSIST_BASE_CACHE and saved.is_file() and saved_sidecar.is_file():
            cache.parent.mkdir(parents=True,exist_ok=True)
            shutil.copy2(saved,cache);shutil.copy2(saved_sidecar,sidecar)
        else:
            execute_command([sys.executable,'-m','esc50_sound','prepare','--root',raw,'--cache',cache], 'prepare_base_cache')
            if PERSIST_BASE_CACHE:
                saved.parent.mkdir(parents=True,exist_ok=True)
                shutil.copy2(cache,saved);shutil.copy2(sidecar,saved_sidecar)
    return cache

def ensure_case_cache(case):
    ensure_audio()
    local=ROOT/'data'/'feature_ablation'
    local.mkdir(parents=True,exist_ok=True)
    path=local/(case+'.npz'); sidecar=local/(case+'.json')
    saved=STORAGE/'feature_caches'/(case+'.npz')
    if not(path.is_file() and sidecar.is_file()):
        if PERSIST_FEATURE_CACHES and saved.is_file() and saved.with_suffix('.json').is_file():
            shutil.copy2(saved,path);shutil.copy2(saved.with_suffix('.json'),sidecar)
        else:
            execute_command([sys.executable,'scripts/feature_ablation.py','prepare','--raw-root',ROOT/'data'/'ESC-50',
                             '--cache-dir',local,'--case',case], f'prepare_{case}')
            if PERSIST_FEATURE_CACHES:
                saved.parent.mkdir(parents=True,exist_ok=True)
                shutil.copy2(path,saved);shutil.copy2(sidecar,saved.with_suffix('.json'))
    return local

if EXECUTE_SETUP:
    print('Data will be prepared lazily when the first selected task needs it.')
else:
    print('No download in plan mode.')

## 5 · Controlled task execution and resumability

Runs never silently overwrite a nonempty result directory. Standard CNN/CRNN/Attention tasks use the existing epoch-boundary `resume-train` when an interrupted `last.pt` is present and compatible; the feature-ablation track **cannot resume** and requires inspection or a fresh run ID. A failed task stops the batch. Development-only robustness uses the selected model checkpoint, but never evaluates the test fold.

In [ ]:
import pickle
BASE_RUNS=RUNS if EXECUTE_SETUP else None

def output_for(task):
    suffix=f'__pilot{PILOT_EPOCHS}ep' if RUN_MODE=='pilot' else ''
    return BASE_RUNS/task['group']/(task['id']+suffix)

def expected_supervised(task, epochs):
    aug=task.get('augmentation','clean')
    return dict(model=task['model'],seed=task['seed'],epochs=epochs,
                batch_size=BATCH_SIZE,lr=LEARNING_RATE,scheduler='cosine',
                train_folds='1,2,3',val_fold=4,test_fold=5,official_folds=False,
                specaugment=aug=='specaugment',noise_probability=0.5 if aug=='noise' else 0.0,
                noise_snr_min_db=10.0,noise_snr_max_db=30.0,
                mixup_alpha=0.4 if aug=='mixup' else 0.0)

def compatible_supervised(task, output, epochs):
    record=json.loads((output/'run.json').read_text())
    expected=expected_supervised(task,epochs)
    if any(record.get('config',{}).get(k) != v for k,v in expected.items()):return False
    from esc50_sound.records import sha256, package_source_identity
    cache=ROOT/'data'/'esc50_features.npz'
    if not cache.is_file() or record.get('cache_sha256') != sha256(cache):return False
    if record.get('environment',{}).get('code_sha256') != package_source_identity()['code_sha256']:return False
    return True

def validation_record_matches(output, metrics, expected_names=None):
    import csv
    from esc50_sound.evaluation import compute_metrics
    rows=list(csv.DictReader((output/'validation_predictions.csv').open()))
    names=[r['filename'] for r in rows]
    if len(rows)!=398 or len(set(names))!=398:return False
    if expected_names is not None and names!=expected_names:return False
    measured=compute_metrics([int(r['target']) for r in rows],[int(r['prediction']) for r in rows])
    return all(abs(measured[k]-metrics[k])<1e-10 for k in ('n','accuracy','macro_f1'))

def completed(task, output, epochs):
    """Check requested science identity and saved evidence before skipping a run."""
    try:
        if task['kind']=='supervised':
            if not (output/'run.json').is_file() or not compatible_supervised(task,output,epochs):return False
            record=json.loads((output/'run.json').read_text())
            if record.get('status')!='completed' or record.get('epochs_recorded')!=epochs:return False
            from esc50_sound.checkpoints import load_checkpoint,validate_checkpoint
            checkpoint=load_checkpoint(output/'best.pt');validate_checkpoint(checkpoint)
            if checkpoint.get('manifest_id') != record.get('manifest_id'):return False
            if checkpoint.get('seed') != task['seed'] or checkpoint.get('model_name') != task['model']:return False
            history=json.loads((output/'history.json').read_text())
            if [r.get('epoch') for r in history] != list(range(1,epochs+1)):return False
            metrics=json.loads((output/'validation.json').read_text())
            from esc50_sound.models import build_model
            build_model(task['model']).load_state_dict(checkpoint['model'],strict=True)
            return validation_record_matches(output,metrics,record['split']['validation']['names'])
        if task['kind']=='feature_ablation':
            from esc50_sound.feature_ablation import by_name,completed_run
            from dataclasses import asdict
            cfg,metrics=completed_run(output)
            expected=dict(spec=asdict(by_name(task['case'])),model=task['model'],seed=task['seed'],epochs=epochs,
                          batch_size=BATCH_SIZE,learning_rate=LEARNING_RATE,train_folds=[1,2,3],
                          validation_fold=4,test_fold=5,source_strict=True)
            return (all(cfg.get(k)==v for k,v in expected.items())
                    and cfg.get('data_complete') is True and metrics.get('n')==398)
        if task['kind']=='baseline':
            from esc50_sound.quality import baseline_artifact_contract,cepstral_artifact_contract
            from esc50_sound.records import sha256
            manifest=json.loads((output/'model.json').read_text())
            if manifest.get('cache_sha256')!=sha256(ROOT/'data'/'esc50_features.npz'):return False
            if (manifest.get('train_folds')!=[1,2,3] or manifest.get('val_fold')!=4
                    or manifest.get('test_fold')!=5 or manifest.get('source_policy')!='source_strict'):return False
            audit=baseline_artifact_contract(output) if task['method']=='logistic' else cepstral_artifact_contract(output)
            return audit['valid']
        if task['kind']=='robustness':
            from esc50_sound.records import sha256
            report=json.loads((output/'stress.json').read_text())
            source=next(t for t in TASKS if t['id']==task['checkpoint_task'])
            return (report.get('split')=='validation' and report.get('seed')==2026
                    and report.get('checkpoint_sha256')==sha256(output_for(source)/'best.pt')
                    and set(task['conditions'])==set(report.get('conditions',{}))
                    and all(m.get('n')==398 for m in report['conditions'].values()))
    except (OSError,ValueError,KeyError,TypeError,RuntimeError,EOFError,pickle.UnpicklingError):
        return False
    return False

def execute_task(task):
    epochs=PILOT_EPOCHS if RUN_MODE=='pilot' else FULL_EPOCHS
    out=output_for(task)
    if completed(task,out,epochs):
        print('SKIP completed:',task['id']);return 'skipped'
    out.parent.mkdir(parents=True,exist_ok=True)
    kind=task['kind']
    raw=ROOT/'data'/'ESC-50'
    base=ROOT/'data'/'esc50_features.npz'
    args=[sys.executable]
    if kind=='feature_ablation':
        case=task['case']; cache_dir=ensure_case_cache(case)
        if out.exists():raise RuntimeError(f'Incomplete ablation run cannot resume: {out}. Inspect or archive it manually.')
        # Existing tool decides output from results-dir / model / case / seedN.
        # Choose an enclosing group-specific results directory, not task directory.
        parent=RUNS/task['group']/('pilot'+str(PILOT_EPOCHS) if RUN_MODE=='pilot' else 'main')
        output=parent/task['model']/case/('seed'+str(task['seed']))
        if completed(task,output,epochs):
            print('SKIP completed:',task['id']);return 'skipped'
        if output.exists():raise RuntimeError(f'Incomplete feature run; inspect before rerunning: {output}')
        args += ['scripts/feature_ablation.py','train','--raw-root',raw,'--cache-dir',cache_dir,
                 '--results-dir',parent,'--case',case,'--seed',task['seed'],'--model',task['model'],
                 '--epochs',epochs,'--batch-size',BATCH_SIZE,'--lr',LEARNING_RATE,
                 '--threads',CPU_THREADS,'--device',DEVICE]
    elif kind=='baseline':
        ensure_base_cache()
        if out.exists():raise RuntimeError(f'Incomplete or incompatible baseline directory: {out}')
        command='baseline' if task['method']=='logistic' else 'cepstral-baseline'
        args += ['-m','esc50_sound',command,'--cache',base,'--output',out]
    elif kind=='supervised':
        ensure_base_cache()
        if out.exists() and (out/'last.pt').is_file() and (out/'run.json').is_file():
            if not compatible_supervised(task,out,epochs):raise RuntimeError(f'Existing run differs from requested task/config/cache/code: {out}')
            print('Attempting compatible epoch-boundary resume:',task['id'])
            args += ['-m','esc50_sound','resume-train','--output',out,'--max-seconds',MAX_SECONDS_PER_TRAIN]
        elif out.exists():
            raise RuntimeError(f'Incomplete run without valid checkpoint: {out}; inspect it first')
        else:
            args += ['-m','esc50_sound','train','--cache',base,'--raw-audio-root',raw,
                     '--model',task['model'],'--output',out,'--epochs',epochs,'--seed',task['seed'],
                     '--scheduler','cosine','--max-seconds',MAX_SECONDS_PER_TRAIN,
                     '--batch-size',BATCH_SIZE,'--lr',LEARNING_RATE,'--threads',CPU_THREADS,'--device',DEVICE]
            aug=task.get('augmentation','clean')
            if aug=='specaugment':args+=['--specaugment']
            if aug=='noise':args+=['--noise-probability',0.5,'--noise-snr-min-db',10,'--noise-snr-max-db',30]
            if aug=='mixup':args+=['--mixup-alpha',0.4]
    elif kind=='robustness':
        ensure_base_cache()
        source=next(t for t in TASKS if t['id']==task['checkpoint_task'])
        source_dir=output_for(source)
        checkpoint=source_dir/'best.pt'
        if not completed(source,source_dir,epochs):
            raise RuntimeError(f'Robustness requires completed checkpoint for {source["id"]}. Run dependency first.')
        if out.exists():raise RuntimeError(f'Incomplete robustness directory: {out}')
        args += ['scripts/legacy/study.py','stress','--checkpoint',checkpoint,'--output',out,
                 '--root',raw,'--cache',base,'--split','validation','--conditions',*task['conditions'],'--seed',2026]
    else:raise NotImplementedError(kind)
    execute_command(args,task['id'])
    expected=output if kind=='feature_ablation' else out
    if not completed(task,expected,epochs):
        raise RuntimeError(f'Task finished subprocess but is not fully completed: {task["id"]}. Inspect {expected}')
    return 'completed'

if not EXECUTE_SETUP:
    print('Execution disabled. Change RUN_MODE to pilot or selected to run tasks.')

## 6 · Run a bounded selection (manual opt-in)

In [ ]:

# Export each verified task atomically so a disconnect does not erase completed evidence.
from datetime import datetime,timezone
EXPORTS=Path('/content/esc50_matrix_exports')
EXPORTS.mkdir(exist_ok=True)
_progress=[]
_original_execute_task=execute_task

def execute_task(task):
    begin=time.monotonic()
    result=_original_execute_task(task)
    if task['kind']=='feature_ablation':
        out=RUNS/task['group']/'main'/task['model']/task['case']/('seed'+str(task['seed']))
    else:out=output_for(task)
    target=EXPORTS/(task['id']+'.zip')
    temporary=target.with_suffix('.tmp')
    with zipfile.ZipFile(temporary,'w',zipfile.ZIP_DEFLATED) as bundle:
        for f in sorted(out.rglob('*')):
            if f.is_file():bundle.write(f,str(f.relative_to(STORAGE)))
        for f in sorted(LOGS.glob(task['id']+'*.log')):bundle.write(f,str(f.relative_to(STORAGE)))
    temporary.replace(target)
    _progress.append(dict(id=task['id'],group=task['group'],kind=task['kind'],status=result,seconds=time.monotonic()-begin,archive=target.name,timestamp_utc=datetime.now(timezone.utc).isoformat()))
    temporary=EXPORTS/'progress.tmp'
    temporary.write_text(json.dumps(_progress,indent=2))
    temporary.replace(EXPORTS/'progress.json')
    print('EXPORTED',task['id'],'|',len(_progress),'/',len(SELECTED),flush=True)
    return result


In [ ]:
if RUN_MODE=='plan':
    print('PLAN PREVIEW ONLY: no task executed.')
else:
    started=time.monotonic()
    _session_deadline=started+60*MAX_SESSION_MINUTES
    count=0
    for task in SELECTED:
        if count>=MAX_TASKS_PER_SESSION:
            print('Session task cap reached. Rerun later to continue.');break
        if (time.monotonic()-started)/60>=MAX_SESSION_MINUTES:
            print('Session time budget reached; stopping before next task.');break
        result=execute_task(task)
        if result!='skipped':count+=1
    print('Newly completed tasks:',count,'| selected tasks:',len(SELECTED))

## 7 · Results (development-only) and optional charts

Results are read from completed `validation.json` files. Empty cells mean **no experiment was run**, not 0% accuracy. To compare sampling rate or feature choices, execute three seeds per case and use the project's `summary` / `contrasts` functions; remember the lightweight ablation uses `cnn`, not the historical `cnn_freq`.

In [ ]:
if not EXECUTE_SETUP:
    print('No results loaded in plan mode.')
else:
    import pandas as pd
    results=[]
    for task in TASKS:
        out=output_for(task)
        if task['kind']=='feature_ablation':
            epochs=PILOT_EPOCHS if RUN_MODE=='pilot' else FULL_EPOCHS
            out=RUNS/task['group']/('pilot'+str(PILOT_EPOCHS) if RUN_MODE=='pilot' else 'main')/task['model']/task['case']/('seed'+str(task['seed']))
        metrics=out/'validation.json'
        if not metrics.is_file() or not completed(task,out,PILOT_EPOCHS if RUN_MODE=='pilot' else FULL_EPOCHS):continue
        record=json.loads(metrics.read_text())
        if 'accuracy' not in record:continue
        results.append(dict(group=task['group'],id=task['id'],accuracy=record['accuracy'],macro_f1=record.get('macro_f1')))
    frame=pd.DataFrame(results)
    if frame.empty:print('No completed development runs yet.')
    else:
        display(frame.sort_values(['group','id']).reset_index(drop=True))
        import matplotlib.pyplot as plt
        subset=frame[frame.group.isin(['A_baselines','B_architecture','C_augmentation'])].copy()
        if not subset.empty:
            fig,ax=plt.subplots(figsize=(10,min(8,max(3,0.4*len(subset)))))
            ax.barh(subset['id'],subset['accuracy']*100)
            ax.set_xlabel('Development validation accuracy (%)')
            ax.set_title('Completed development experiments — not held-out test')
            ax.invert_yaxis();plt.tight_layout();plt.show()
if EXECUTE_SETUP:
    stress_rows=[]
    for task in TASKS:
        if task['kind']!='robustness':continue
        out=output_for(task)
        if not completed(task,out,PILOT_EPOCHS if RUN_MODE=='pilot' else FULL_EPOCHS):continue
        for condition,metrics in json.loads((out/'stress.json').read_text())['conditions'].items():
            stress_rows.append(dict(task=task['id'],condition=condition,accuracy=metrics['accuracy'],macro_f1=metrics['macro_f1'],n=metrics['n']))
    if stress_rows:display(pd.DataFrame(stress_rows))


## 8 · Feature ablation comparison (after completing D tasks)

The summary and paired-contrast commands only evaluate the development validation fold. Reports are given unique filenames so previous research records are not overwritten. Run this section after completing the relevant cases (three seeds preferred).

In [ ]:
if not EXECUTE_SETUP:
    print('Feature contrast generation disabled in plan mode.')
else:
    data_dir=RUNS/'D_feature_sampling'/('pilot'+str(PILOT_EPOCHS) if RUN_MODE=='pilot' else 'main')
    if list(data_dir.rglob('validation.json')) if data_dir.exists() else []:
        from datetime import datetime,timezone
        timestamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
        execute_command([sys.executable,'scripts/feature_ablation.py','summary','--results-dir',data_dir,
                         '--output',REPORTS/f'feature_summary_{timestamp}.csv'],'feature_summary')
        execute_command([sys.executable,'scripts/feature_ablation.py','contrasts','--results-dir',data_dir,
                         '--output',REPORTS/f'feature_contrasts_{timestamp}.csv'],'feature_contrasts')
    else:print('No completed feature ablations to aggregate.')

## 9 · Scope and honest completion criteria

- **Data integrity:** official metadata, source-strict splits, label alignment and train-only normalization are retained. No redundant new SHA-256 verification pipeline is added to the feature-ablation track; the old cache/resume routes maintain compatibility.
- **No automatic test:** test fold 5 is never used by this notebook. Existing historical test exposure cannot be undone; do not claim independent confirmation from development scores.
- **Colab interruptions:** a completed run is skipped; an incomplete main training run is resumed only when the existing strict environment/data checks accept it. Feature-ablation runs do not resume from mid-training.
- **Storage:** dataset ZIP/processed features and results can consume substantial Google Drive space. Keep local caches optional and archive results after each stage.
- **Future models:** PANNs, CLAP, AST, BEATs and RL are recorded as future work, *not runnable in this notebook*. Do not fabricate their accuracy.
- **Training budget:** 15 architecture + 9 augmentation + 36 feature runs + 2 traditional baselines + 3 optional validation stress tests = **65 registered executable tasks**, excluding pilots. A study does **not** require running all 65; choose the research questions and compute budget deliberately.